In [1]:
import sys, importlib
MODULE_DIR = "/LARGE0/gr10478/b37974/Pulmonary_Hypertension/cteph_agp3k/review_analysis/script"
if MODULE_DIR not in sys.path:
    sys.path.insert(0, MODULE_DIR)

import fine_map_susie_tools
importlib.reload(fine_map_susie_tools)

<module 'fine_map_susie_tools' from '/LARGE0/gr10478/b37974/Pulmonary_Hypertension/cteph_agp3k/review_analysis/script/fine_map_susie_tools.py'>

In [ ]:
from fine_map_susie_tools import build_lead_and_locus_tables

summary_json = build_lead_and_locus_tables(
    plink_stats_file="/LARGE0/gr10478/b37974/Pulmonary_Hypertension/cteph_agp3k/analysis/assoc_plink2/results/01.assoc_result/additive/cteph_agp3k.sex.10pc.additive.PHENO1.glm.logistic",
    range_bp=1_000_000,
    windowsizekb=500,
    sig_level=5e-8,
    output_prefix="cteph_agp3k.lowfreq_common"
)

In [ ]:
from fine_map_susie_tools import build_ld_matrices_from_summary

ld_json = build_ld_matrices_from_summary(
    summary_json_path=summary_json,
    bed_prefix='/LARGE0/gr10478/b37974/Pulmonary_Hypertension/cteph_agp3k/wgs/19.tommo_panel_filter/cteph_agp3k.lowfreq_common', 
    output_prefix="cteph_agp3k.lowfreq_common",
    sample_mode="ctrl", # "all", "case", "ctrl"
    case_prefix="PHOM"
)

In [ ]:
from fine_map_susie_tools import plot_ld_heatmaps_from_index

pdf_path = plot_ld_heatmaps_from_index(
    index_json_path=ld_json,
    dpi=150,
    output_prefix="cteph_agp3k.lowfreq_common",
    show_lead_cross=True
)

In [ ]:
import json
import pandas as pd

with open(summary_json, 'r') as f:
    summary_data = json.load(f)

lead_tsv_path = summary_data["outputs"]["lead_tsv"]
lead_df = pd.read_csv(lead_tsv_path, sep="\t")
lead_variant_ls = lead_df["SNPID"].tolist()

In [ ]:
sample_size = 2644 #ctrl model
base_path = "/LARGE0/gr10478/b37974/Pulmonary_Hypertension/cteph_agp3k/review_analysis/05.fine_map_susie/tmp"
susie_out_dir = "/LARGE0/gr10478/b37974/Pulmonary_Hypertension/cteph_agp3k/review_analysis/05.fine_map_susie/susie_result"
pip_threshold = 0.000

In [ ]:
import subprocess
import os
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed
import time
from datetime import datetime

# 清除之前的输出（防止重复运行导致的混乱）
print("=" * 60)
print(f"SuSiE 并行分析开始 - {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 60)

# 确保输出目录存在
os.makedirs(susie_out_dir, exist_ok=True)

# susie_cli.r 脚本路径
susie_cli_path = os.path.join(MODULE_DIR, "susie_cli.r")

# 定义单个变异分析函数
def run_susie_analysis(args):
    """运行单个变异的SuSiE分析"""
    lead_variant, sample_size, base_path, susie_out_dir, pip_threshold = args

    try:
        # 构建命令
        cmd = [
            "Rscript", susie_cli_path,
            "-s", str(sample_size),
            "-l", lead_variant,
            "-b", base_path,
            "-o", susie_out_dir,
            "-p", str(pip_threshold),
            "-v"  # 启用详细输出
        ]
        
        start_time = time.time()
        
        # 运行命令
        result = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            timeout=600  # 10分钟超时
        )
        
        end_time = time.time()
        duration = end_time - start_time
        
        return {
            'variant': lead_variant,
            'success': result.returncode == 0,
            'returncode': result.returncode,
            'stdout': result.stdout,
            'stderr': result.stderr,
            'duration': duration,
            'command': ' '.join(cmd)
        }
        
    except subprocess.TimeoutExpired:
        return {
            'variant': lead_variant,
            'success': False,
            'error': '超时 (>10分钟)',
            'duration': 600,
            'command': ' '.join(cmd) # type: ignore
        }
    except Exception as e:
        return {
            'variant': lead_variant,
            'success': False,
            'error': str(e),
            'duration': 0,
            'command': ' '.join(cmd) # type: ignore
        }

# 检查脚本是否存在
if not os.path.exists(susie_cli_path):
    print(f"❌ 错误: SuSiE CLI脚本不存在: {susie_cli_path}")
else:
    print(f"✅ 找到 SuSiE CLI 脚本: {susie_cli_path}")
    print(f"🚀 开始并行分析 {len(lead_variant_ls)} 个变异 (使用3个进程)")
    print()
    
    # 准备参数列表
    args_list = [
        (variant, sample_size, base_path, susie_out_dir, pip_threshold) 
        for variant in lead_variant_ls
    ]
    
    # 使用ProcessPoolExecutor进行3核心并行计算
    successful_count = 0
    failed_count = 0
    results_list = []
    
    start_total = time.time()
    
    with ProcessPoolExecutor(max_workers=3) as executor:
        # 提交所有任务
        future_to_args = {
            executor.submit(run_susie_analysis, args): args[0] 
            for args in args_list
        }
        
        # 处理完成的任务
        completed_variants = set()  # 用于防止重复输出
        
        for future in as_completed(future_to_args):
            result = future.result()
            variant = result['variant']
            
            # 防止重复处理同一个变异
            if variant in completed_variants:
                continue
            completed_variants.add(variant)
            
            results_list.append(result)
            current_count = len(completed_variants)
            
            print(f"📊 进度: {current_count}/{len(lead_variant_ls)} | {variant}")
            
            if result['success']:
                print(f"   ✅ 分析成功 (耗时: {result['duration']:.1f}秒)")
                successful_count += 1
            else:
                print(f"   ❌ 分析失败: {result.get('error', '未知错误')}")
                failed_count += 1
                if result.get('stderr'):
                    # 只显示错误信息的前200字符
                    error_msg = result['stderr'][:200].replace('\n', ' ')
                    print(f"   📝 错误详情: {error_msg}...")
            print()
    
    end_total = time.time()
    total_duration = end_total - start_total
    
    print("=" * 60)
    print("📈 并行分析完成统计")
    print("=" * 60)
    print(f"⏱️  总耗时: {total_duration:.1f} 秒")
    print(f"📁 总变异数量: {len(lead_variant_ls)}")
    print(f"✅ 成功分析: {successful_count}")
    print(f"❌ 失败分析: {failed_count}")
    print(f"📊 成功率: {successful_count/len(lead_variant_ls)*100:.1f}%")
    print(f"💾 结果保存在: {susie_out_dir}")
    print("=" * 60)

In [ ]:
import os
import json
from pathlib import Path

def create_susie_summary(lead_variant_ls, out_dir):
    """
    根据主导变异列表创建 SuSiE 结果汇总文件。

    参数：
        lead_variant_ls (list): 主导变异 ID 列表（形如 'chr3:154069965:A:G'）
        out_dir (str): 输出目录

    返回：
        str: 汇总 JSON 文件路径
    """
    out_dir = Path(out_dir)
    susie_summary = {}

    print("=== 创建 SuSiE 结果汇总文件 ===")

    for lead_variant in lead_variant_ls:
        formatted_variant = lead_variant.replace(":", "_")

        json_file = out_dir / f"{formatted_variant}.susie_results.json"
        log_file = out_dir / f"{formatted_variant}.susie_analysis.log"

        json_exists = json_file.exists()
        log_exists = log_file.exists()

        susie_summary[lead_variant] = {
            "json_file": str(json_file) if json_exists else None,
            "log_file": str(log_file) if log_exists else None,
            "json_exists": json_exists,
            "log_exists": log_exists,
            "formatted_id": formatted_variant
        }

        status = []
        status.append("JSON✓" if json_exists else "JSON✗")
        status.append("LOG✓" if log_exists else "LOG✗")

        print(f"{lead_variant}: {' '.join(status)}")

    # 保存汇总文件
    summary_file = out_dir / "susie_summary.json"
    with open(summary_file, "w", encoding="utf-8") as f:
        json.dump(susie_summary, f, indent=2, ensure_ascii=False)

    print(f"\n汇总文件已保存: {summary_file}")

    # 打印统计信息
    total_variants = len(lead_variant_ls)
    successful_json = sum(1 for v in susie_summary.values() if v["json_exists"])
    successful_log = sum(1 for v in susie_summary.values() if v["log_exists"])

    print("\n=== 统计信息 ===")
    print(f"总变异数量: {total_variants}")
    print(f"成功生成JSON文件: {successful_json}/{total_variants}")
    print(f"成功生成LOG文件: {successful_log}/{total_variants}")

    # ✅ 返回汇总 JSON 路径
    susie_json_path = str(summary_file)
    return susie_json_path

In [ ]:
susie_json_path = create_susie_summary(lead_variant_ls, susie_out_dir)
print("SuSiE汇总返回路径：", susie_json_path)

In [ ]:
from fine_map_susie_tools import integrate_susie_results_with_sumstat

susie_summary_outdir = "/LARGE0/gr10478/b37974/Pulmonary_Hypertension/cteph_agp3k/review_analysis/05.fine_map_susie/susie_summary"
integrated_result = integrate_susie_results_with_sumstat(
    susie_summary_json_path=susie_json_path,
    ld_matrices_summary_json_path=ld_json,
    output_dir=susie_summary_outdir,
    output_prefix="susie_integrated"
)

In [2]:
from fine_map_susie_tools import plot_manhattan_plots_from_integrated_results

pdf_path = plot_manhattan_plots_from_integrated_results(
    integrated_json_path="/LARGE0/gr10478/b37974/Pulmonary_Hypertension/cteph_agp3k/review_analysis/05.fine_map_susie/susie_summary/susie_integrated.integrated_results.json",
    output_prefix="susie",
    highlight_credible_sets=True,
    dpi=300
)

[plot_manhattan_plots] 读取整合结果: /LARGE0/gr10478/b37974/Pulmonary_Hypertension/cteph_agp3k/review_analysis/05.fine_map_susie/susie_summary/susie_integrated.integrated_results.json
[plot_manhattan_plots] 输出PDF路径: /LARGE0/gr10478/b37974/Pulmonary_Hypertension/cteph_agp3k/review_analysis/05.fine_map_susie/susie_summary/susie.manhattan_plots.pdf
[plot_manhattan_plots] 处理 lead variant: chr3:154069965:A:G
[plot_manhattan_plots] 警告: 跳过 chr3:154069965:A:G，增强汇总统计文件不存在
[plot_manhattan_plots] 处理 lead variant: chr16:53887925:T:C


/LARGE0/gr10478/b37974/Pulmonary_Hypertension/cteph_agp3k/review_analysis/script/fine_map_susie_tools.py:1528: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0, 0.98, 0.95])  # 为suptitle和colorbar留出空间


[plot_manhattan_plots] 完成 chr16:53887925:T:C: 1933 个变体
[plot_manhattan_plots] 处理 lead variant: chr17:13528059:G:A


/LARGE0/gr10478/b37974/Pulmonary_Hypertension/cteph_agp3k/review_analysis/script/fine_map_susie_tools.py:1528: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0, 0.98, 0.95])  # 为suptitle和colorbar留出空间


[plot_manhattan_plots] 完成 chr17:13528059:G:A: 2824 个变体
[plot_manhattan_plots] 曼哈顿图已保存到: /LARGE0/gr10478/b37974/Pulmonary_Hypertension/cteph_agp3k/review_analysis/05.fine_map_susie/susie_summary/susie.manhattan_plots.pdf


In [ ]:
# from fine_map_susie_tools import plot_manhattan_plots_from_integrated_results

# pdf_path = plot_manhattan_plots_from_integrated_results(
#     integrated_json_path=integrated_result["integrated_json_path"],
#     output_prefix="susie",
#     highlight_credible_sets=True,
#     dpi=300
# )